# SecureFace Restructured Notebook

This notebook provides a modular and extensible pipeline for baseline face detection, adversarial attacks (FGSM), GAN-based defense, and comprehensive evaluation.

**Original notebook remains unchanged!**

## 1. Environment Setup & Dependencies
*Pin versions for reproducibility.*

In [ ]:
# Install required libraries (pinned versions)
!pip install ultralytics==8.4.0 gfpgan==1.3.8 basicsr==1.4.2 facexlib==0.3.0
# Fix torchvision error for basicsr
!sed -i "s/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/g" /usr/local/lib/python3.*/dist-packages/basicsr/archs/rrdbnet_arch.py


In [ ]:
import os
import numpy as np
import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from gfpgan import GFPGANer
from google.colab.patches import cv2_imshow
from google.colab import files
import random


## 2. Utilities & Helper Functions

In [ ]:
def load_yolo():
    # Tries to load face-specific, then generic
    try:
        model = YOLO('yolov8n-face.pt')
    except Exception as e:
        print('Falling back to yolov8n.pt')
        model = YOLO('yolov8n.pt')
    return model

def load_gfpgan():
    model_url = 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth'
    return GFPGANer(model_path=model_url, upscale=1, arch='clean', channel_multiplier=2, bg_upsampler=None)

def show_img(img, title=None):
    if img is None: print('Image is None'); return
    # CV2 uses BGR, pyplot uses RGB
    plt.figure(figsize=(8,6))
    plt.axis('off')
    if title: plt.title(title)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.show()

def draw_boxes(img, boxes, confidences=None, color=(0,255,0), label=None):
    boxed = img.copy()
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)        cv2.rectangle(boxed, (x1, y1), (x2, y2), color, 2)
        if confidences is not None:
            cv2.putText(boxed, f'Conf: {confidences[i]:.2f}', (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
        if label:
            cv2.putText(boxed, f'{label}', (x1, y2+20), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
    return boxed


## 3. Image Upload / Selection
*Upload or select an image to begin the pipeline.*

In [ ]:
uploaded = files.upload()
img_path = next(iter(uploaded)) # just take the first uploaded
img = cv2.imread(img_path)
if img is None:
    raise FileNotFoundError('Failed to load image. Upload a .jpg or .png file.')
print(f'Image loaded: {img_path}, shape={img.shape}')
show_img(img, title='Original Image')


## 4. Load Models
*YOLO for detection, GFPGAN for defense.*

In [ ]:
detector = load_yolo()
purifier = load_gfpgan()
print('YOLO & GFPGAN loaded.')


## 5. Baseline (No Attack) Detection

In [ ]:
def detect_faces(detector, img):
    results = detector(img, verbose=False)
    boxes = results[0].boxes.xyxy.cpu().numpy() if len(results[0].boxes) else []
    confs = results[0].boxes.conf.cpu().numpy() if len(results[0].boxes) else []
    return boxes, confs

# Detect
baseline_boxes, baseline_conf = detect_faces(detector, img)
img_baseline = draw_boxes(img, baseline_boxes, baseline_conf, color=(0,255,0), label='Baseline')
show_img(img_baseline, title='Baseline YOLO Detection')
print(f'Faces detected: {len(baseline_boxes)}; Confidences: {baseline_conf}')


## 6. Adversarial Attack (FGSM)
*Perturb image using Fast Gradient Sign Method to fool YOLO detector.*

**Note:** FGSM is generally designed for classification, but for demo purposes, we will treat YOLO as a face classifier on a detected bounding box.

In [ ]:
def fgsm_attack(img, model, epsilon=8/255):
    # Convert to torch tensor
    im = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    t = torch.tensor(im, dtype=torch.float).permute(2,0,1)/255.0
    t = t.unsqueeze(0).to('cuda' if torch.cuda.is_available() else 'cpu').requires_grad_(True)
    # Use a dummy target and loss; YOLO is not usually differentiable in deployment, this is a simplification.
    yolo = model
    out = yolo(t)[0]  # forward
    if hasattr(out, 'boxes') and len(out.boxes) > 0:
        loss = -out.boxes.conf.mean() # Adversarial: try to reduce average confidence
        loss.backward()
        data_grad = t.grad.data
        perturbed = t + epsilon * data_grad.sign()
        perturbed = torch.clamp(perturbed, 0, 1)
        pert = perturbed.squeeze().permute(1,2,0).cpu().detach().numpy()*255
        img_p = cv2.cvtColor(pert.astype(np.uint8), cv2.COLOR_RGB2BGR)
        return img_p
    else: # If no faces, just return original
        print('No faces for adversarial attack, returning original image.')
        return img


In [ ]:
img_adv = fgsm_attack(img, detector, epsilon=8/255)
show_img(img_adv, title='Adversarial Image (FGSM)')
# Detect
adv_boxes, adv_conf = detect_faces(detector, img_adv)
img_adv_detect = draw_boxes(img_adv, adv_boxes, adv_conf, color=(0,0,255), label='Attack')
show_img(img_adv_detect, title='Detection after Attack')
print(f'Attack: Faces detected: {len(adv_boxes)}; Confidences: {adv_conf}')


## 7. Defense Step: Apply GAN Purification

In [ ]:
defense_img, _, _ = purifier.enhance(img_adv, has_aligned=False, only_center_face=False, paste_back=True)
show_img(defense_img, title='Purified Image (GFPGAN)')
# Detect
def_boxes, def_conf = detect_faces(detector, defense_img)
img_def_detect = draw_boxes(defense_img, def_boxes, def_conf, color=(255,140,0), label='Defense')
show_img(img_def_detect, title='Detection after Defense (GFPGAN)')
print(f'Defense: Faces detected: {len(def_boxes)}; Confidences: {def_conf}')


## 8. Evaluation Metrics & Comparison

In [ ]:
def detection_accuracy(num_detected, total_true):
    return num_detected / max(total_true, 1)

def average_conf(conf):
    return np.mean(conf) if len(conf) else 0.0

def l2_perturbation(img1, img2):
    return np.linalg.norm(img1.astype(np.float32)-img2.astype(np.float32))/img1.size

metrics = pd.DataFrame([
    {
        'Step': 'Baseline',
        'Faces': len(baseline_boxes),
        'AvgConf': average_conf(baseline_conf),
        'L2Pert': 0
    },
    {
        'Step': 'Attack',
        'Faces': len(adv_boxes),
        'AvgConf': average_conf(adv_conf),
        'L2Pert': l2_perturbation(img, img_adv),
    },
    {
        'Step': 'Defense',
        'Faces': len(def_boxes),
        'AvgConf': average_conf(def_conf),
        'L2Pert': l2_perturbation(img_adv, defense_img)
    },
])display(metrics)


## 9. Conclusion
You can now experiment by uploading different images, adjusting `epsilon` value in the attack, or measuring additional security/AI metrics!

---
**This notebook was generated from code. Please verify, adapt attack/defense for full research usage, and cite the appropriate YOLO/GFPGAN authors if used in publications.**